In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

In [ ]:
# 1. بارگذاری داده‌ها
train_df = pd.read_csv(r'train.csv')
test_df = pd.read_csv(r'test.csv')

print("داده‌های آموزشی بارگذاری شدند.")
print(f"شکل (Shape) مجموعه داده آموزش: {train_df.shape}")
print(f"شکل (Shape) مجموعه داده تست: {test_df.shape}\n")

In [ ]:
# 2. جدا کردن ویژگی‌ها (X) و متغیر هدف (y)
feature_columns = [col for col in train_df.columns if col != 'electricity_consumption_kWh']
X_train_full = train_df[feature_columns]
y_train_full = train_df['electricity_consumption_kWh']

print(f"ویژگی‌های آموزشی استخراج شدند: {feature_columns}")
print("متغیر هدف آموزشی استخراج شد: 'electricity_consumption_kWh'")

In [ ]:
# 3. تعریف Pipeline (مقیاس‌ساز + مدل) برای جلوگیری از نشت داده
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        max_iter=5000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=20,
        alpha=0.0001,
        tol=1e-4,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-8
    ))
])

print("\nPipeline شامل StandardScaler و MLPRegressor ساخته شد.")


In [ ]:
# 4. ارزیابی با Cross-Validation (بدون نشت داده)
print("\n--- ارزیابی مدل با Cross-Validation ---")
cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(mlp_pipeline, X_train_full, y_train_full,
                         cv=cv_kfold, scoring='r2', n_jobs=-1)

print(f"امتیازات R² برای هر Fold: {np.round(scores, 4)}")
print(f"میانگین امتیاز R²: {scores.mean():.4f}")
print(f"انحراف معیار امتیازات R²: {scores.std():.4f}")


In [ ]:
# 5. آموزش نهایی روی کل داده‌های آموزشی
print("\n--- آموزش نهایی مدل روی کل داده‌های آموزشی ---")
mlp_pipeline.fit(X_train_full, y_train_full)
print("مدل نهایی آموزش دید.")

In [ ]:
# 6. پیش‌بینی روی داده‌های تست (Pipeline خودکار تست را مقیاس می‌کند)
print("\n--- پیش‌بینی روی داده‌های تست ---")
y_test_pred = mlp_pipeline.predict(test_df[feature_columns])
print("پیش‌بینی‌ها انجام شد.")

In [ ]:
# 7. ساخت فایل خروجی
submission = pd.DataFrame({'electricity_consumption_kWh': y_test_pred})
submission_filename = 'submission_neural_network_final.csv'
submission.to_csv(submission_filename, index=False)

print(f"\nفایل '{submission_filename}' با موفقیت ایجاد شد.")
print("پیش‌نمایش ۵ سطر اول:")
print(submission.head())

In [ ]:
import joblib

# استخراج و ذخیره مستقیم مدل شبکه عصبی (MLP) از دل پایپ‌لاین
extracted_mlp = mlp_pipeline.named_steps['mlp']
joblib.dump(extracted_mlp, 'mlp_model.joblib')

print("✅ مدل شبکه عصبی (MLP) با موفقیت از پایپ‌لاین استخراج و با نام mlp_model.joblib ذخیره شد!")